# Prompt Engine Usage Examples

This notebook demonstrates the core functionality of the Prompt Engine system.

## Available Prompt Components for EDA

The system provides modular components that represent different stages of the EDA pipeline:

In [1]:
# Import all available EDA component enums
# These represent different stages of the EDA pipeline, each with multiple strategy options
from prompt_engine.components.eda import (
    DataIngestionComponent,          # Handles data loading: CSV, EXCEL, PARQUET, JSON
    MissingValuesComponent,          # Missing value strategies: IMPUTE_MEAN, DROP_ROWS, IMPUTE_KNN, etc.
    TypeHandlingComponent,           # Data type management: INFER_TYPES, CAST_MANUALLY, PARSE_DATETIME, etc.
    UnivariativeAnalysisComponent,   # Single feature analysis: DESCRIPTIVE_STATS
    BivariativeAnalysisComponent,    # Two-feature relationship analysis: PLACEHOLDER
    OutliersDetectionComponent,      # Outlier detection methods: Z_SCORE, IQR, ISOLATION_FOREST, etc.
    CardinalityComponent,            # Cardinality analysis: UNIQUE_RATIO, TAG_LOW_CARDINALITY, etc.
    TargetAnalysisComponent,         # Target variable analysis: CLASS_IMBALANCE, FEATURE_TARGET_RELATIONSHIP, etc.
    DataQualityComponent             # Data quality checks: DUPLICATE_ROWS, INVALID_RANGES, NULL_LIKE_VALUES
    )

## Available Templates for EDA

Templates provide pre-configured sequences of components for common EDA workflows:

In [2]:
# Import pre-configured EDA templates for common workflows
# Each template defines a specific sequence of components for different EDA scenarios
from prompt_engine.template.eda import (
    FullBasicEDAPipelineTemplate,                  # Standard EDA: ingestion → types → missing values → analysis
    RobustTypeCleaningTemplate,                    # Deep type handling with multiple cleaning steps
    AdvancedMissingValueImputationTemplate,        # Sophisticated missing value handling strategies
    DataValidationAndCleaningTemplate,             # Quality checks and data cleaning workflow
    OutlierDetectionAndCleanupTemplate,            # Comprehensive outlier detection pipeline
    CardinalityAnalysisTemplate,                   # Feature cardinality assessment workflow
    TargetDistributionAndRelationshipTemplate,     # Target variable analysis pipeline
    FeatureEngineeringStarterTemplate,             # Feature preparation for modeling
    RegressionTargetBinningTemplate,               # Continuous target binning workflow
    MinimalQuickLookTemplate,                      # Rapid dataset overview
    CleanAndDescribeTemplate                       # Basic cleaning and description workflow
    
)

## Available Composer

The PromptComposer allows flexible composition of prompts without predefined ordering:

In [3]:
# Import the flexible PromptComposer for dynamic prompt composition
# Unlike templates, composer allows arbitrary component ordering and flexible workflow design
from prompt_engine.composer import PromptComposer

## How to Use Templates

Templates enforce a predefined component order and provide type-safe prompt composition.

### Step 1: Select the Most Suitable Template

Choose a template that matches your EDA workflow requirements:

In [4]:
# Example: Select the FullBasicEDAPipelineTemplate for standard EDA workflow
# This template provides a comprehensive yet straightforward approach to data exploration
FullBasicEDAPipelineTemplate

prompt_engine.template.eda.FullBasicEDAPipelineTemplate

### Step 2: Check Component Order

Examine the predefined component sequence for your chosen template:

In [5]:
# Inspect the predefined component sequence for the selected template
# This shows the exact order of components that must be followed when providing strategies
FullBasicEDAPipelineTemplate.get_components_order()

[prompt_engine.registry.eda.data_ingestion.DataIngestionPromptRegistry,
 prompt_engine.registry.eda.type_handling.TypeHandlingPromptRegistry,
 prompt_engine.registry.eda.missing_values.MissingValuesPromptRegistry,
 prompt_engine.registry.eda.univariative_analysis.UnivariativeAnalysisPromptRegistry]

### Step 3: Generate Composed Prompt

Provide strategy enum values for each component in the same order as shown above:

In [6]:
# Generate a composed prompt using the template with specific strategy choices
# The strategy list must match the component order shown above exactly
prompt = FullBasicEDAPipelineTemplate.get_prompt(strategies=[
    DataIngestionComponent.CSV,                    # 1st: Load data from CSV file
    TypeHandlingComponent.INFER_TYPES,             # 2nd: Automatically infer optimal data types
    MissingValuesComponent.IMPUTE_MEAN,             # 3rd: Handle missing values using KNN imputation
    UnivariativeAnalysisComponent.DESCRIPTIVE_STATS # 4th: Generate descriptive statistics
])

# Display the composed prompt - each component contributes its specific instruction
print(prompt)

DataIngestion.CSV: placeholder


TypeHandling.INFER_TYPES: placeholder


MissingValues.IMPUTE_MEAN: placeholder


UnivariativeAnalysis.DESCRIPTIVE_STATS: placeholder



### Template Validation

⚠️ **Warning**: Templates enforce strict component ordering and type validation. Providing strategies in the wrong order or with invalid types will result in errors:

In [8]:
# DEMONSTRATION: Template validation - wrong component order
# This example shows what happens when strategies are provided in incorrect order
prompt = FullBasicEDAPipelineTemplate.get_prompt(strategies=[
    DataIngestionComponent.CSV,                    # ✓ Correct position (1st)
    MissingValuesComponent.IMPUTE_KNN,             # ✗ Wrong position (should be 3rd, not 2nd)
    TypeHandlingComponent.INFER_TYPES,             # ✗ Wrong position (should be 2nd, not 3rd)  
    UnivariativeAnalysisComponent.DESCRIPTIVE_STATS # ✓ Correct position (4th)
])
print(prompt)

TypeError: MissingValuesComponent.IMPUTE_KNN is unexpected strategy for TypeHandlingComponent

In [10]:
# DEMONSTRATION: Template validation - invalid component type
# This example shows what happens when non-enum values are provided as strategies
prompt = FullBasicEDAPipelineTemplate.get_prompt(strategies=[
    DataIngestionComponent.CSV,                    # ✓ Valid component enum
    TypeHandlingComponent.INFER_TYPES,             # ✓ Valid component enum
    1,                                             # ✗ Invalid type (integer instead of enum)
    UnivariativeAnalysisComponent.DESCRIPTIVE_STATS # ✓ Valid component enum
])
print(prompt)

TypeError: 1 is unexpected strategy for MissingValuesComponent

## How to Use Composer

The PromptComposer provides more flexibility by allowing custom component ordering and dynamic prompt composition.

### Flexible Component Ordering

With the composer, you can arrange components in any order that suits your workflow:

In [11]:
# DEMONSTRATION: Flexible component ordering with PromptComposer
# Unlike templates, the composer allows arbitrary component sequences and combinations

# Example A: Custom workflow combining data ingestion, type handling, cardinality, and quality checks
prompt_a = PromptComposer.get_prompt(strategies=[
    DataIngestionComponent.EXCEL,                  # Load Excel data
    TypeHandlingComponent.CAST_MANUALLY,           # Manual type casting
    CardinalityComponent.DETECT_ID_LIKE_COLUMNS,   # Identify potential ID columns
    DataQualityComponent.DUPLICATE_ROWS            # Check for duplicate rows
])

# Example B: Minimal single-component workflow
prompt_b = PromptComposer.get_prompt(strategies=[
    DataIngestionComponent.EXCEL                   # Just load Excel data
])

# Example C: Target-focused analysis workflow with mixed component order
prompt_c = PromptComposer.get_prompt(strategies=[
    TargetAnalysisComponent.CLASS_IMBALANCE,       # Analyze target class distribution
    UnivariativeAnalysisComponent.DESCRIPTIVE_STATS, # Generate feature statistics
    MissingValuesComponent.BACKWARD_FILL           # Fill missing values backwards
])

# Display all three composed prompts to show flexibility
print(f"Custom Multi-Component Workflow:\n{prompt_a}\n")
print('-' * 50)
print(f"Minimal Single-Component Workflow:\n{prompt_b}\n")
print('-' * 50)
print(f"Target-Focused Analysis Workflow:\n{prompt_c}\n")

Custom Multi-Component Workflow:
DataIngestion.EXCEL: placeholder

TypeHandling.CAST_MANUALLY: placeholder

Cardinality.DETECT_ID_LIKE_COLUMNS: placehodler

DataQuality.DUPLICATE_ROWS: placeholder

--------------------------------------------------
Minimal Single-Component Workflow:
DataIngestion.EXCEL: placeholder

--------------------------------------------------
Target-Focused Analysis Workflow:
TargetAnalysis.CLASS_IMBALANCE: placeholder

UnivariativeAnalysis.DESCRIPTIVE_STATS: placeholder

MissingValues.BACKWARD_FILL: placeholder



### Composer Validation

⚠️ **Warning**: The composer validates component types and registry availability. Using invalid components or unregistered strategies will result in errors:

In [12]:
# DEMONSTRATION: Composer validation - invalid component type
# The composer validates that all provided strategies are valid enum instances
PromptComposer.get_prompt(strategies=[
    1  # ✗ Invalid: integer instead of component enum
])

TypeError: Component must be an Enum class, got: <class 'int'>

In [13]:
# DEMONSTRATION: Composer validation - unregistered component
# The composer requires that all component types have registered prompt registries

# Define a custom component enum (valid enum structure)
from enum import Enum, auto

class NewComponent(Enum):
    """Custom component not registered with the composer"""
    STRATEGY_1 = auto()
    
# Attempt to use the unregistered component
# This will fail because NewComponent has no registered prompt registry
prompt_b = PromptComposer.get_prompt(strategies=[
    NewComponent.STRATEGY_1  # ✗ Valid enum but no registered prompt registry
])

KeyError: "No prompt registry registered for component: <enum 'NewComponent'>"